# BPE Tokenizer — from scratch

Implementing Byte Pair Encoding from the ground up, in pure Python, no external libraries.

The goal: understand how GPT-4 turns raw text into numbers before any neural network sees it.
At the end I compare my tokenizer against tiktoken (`cl100k_base`) to see how far 50 merges vs 100k merges gets you.

## 1. Corpus

BPE learns from data. I'm using 100 sentences covering everyday English, ML vocabulary and science — so common patterns like `"the"`, `"training"`, `"model"` appear enough times to get merged early.

The more frequent a pair of symbols is across the corpus, the sooner BPE merges it into a single token.

In [ ]:
from collections import defaultdict

corpus = [
    "the cat sat on the mat",
    "the dog ran in the park",
    "a quick brown fox jumps over the lazy dog",
    "she sells seashells by the seashore",
    "how much wood would a woodchuck chuck",
    "to be or not to be that is the question",
    "all that glitters is not gold",
    "the early bird catches the worm",
    "actions speak louder than words",
    "every cloud has a silver lining",
    "better late than never",
    "do not judge a book by its cover",
    "the pen is mightier than the sword",
    "two wrongs do not make a right",
    "when in rome do as the romans do",
    "the grass is always greener on the other side",
    "a penny saved is a penny earned",
    "honesty is the best policy",
    "where there is a will there is a way",
    "time flies when you are having fun",
    "the computer processes data very quickly",
    "software engineers write code every day",
    "the internet connects people around the world",
    "data science combines statistics and programming",
    "open source software is freely available",
    "the database stores millions of records",
    "machine learning models improve with more data",
    "the algorithm runs in linear time",
    "version control helps teams collaborate",
    "the API returns a json response",
    "the model learns from training data",
    "neural networks are inspired by the brain",
    "gradient descent minimizes the loss function",
    "the learning rate controls how fast the model learns",
    "overfitting happens when the model memorizes training data",
    "regularization reduces overfitting in neural networks",
    "the training loop runs for many epochs",
    "batch normalization stabilizes training of deep networks",
    "the validation loss should decrease during training",
    "dropout randomly disables neurons during training",
    "the transformer architecture uses self attention",
    "attention allows the model to focus on relevant tokens",
    "the embedding layer maps tokens to dense vectors",
    "positional encoding tells the model the order of tokens",
    "the softmax function converts logits to probabilities",
    "cross entropy loss measures prediction error",
    "backpropagation computes gradients through the network",
    "the optimizer updates weights to reduce the loss",
    "hyperparameters are set before training begins",
    "the model checkpoint saves weights during training",
    "fine tuning adapts a pretrained model to a new task",
    "transfer learning reuses knowledge from a previous task",
    "the language model predicts the next token",
    "tokenization splits text into smaller units called tokens",
    "the vocabulary contains all possible tokens",
    "byte pair encoding merges the most frequent pairs",
    "the tokenizer encodes text as a list of integers",
    "decoding converts integers back to readable text",
    "the context window limits how much text the model sees",
    "inference is the process of generating text from a model",
    "natural language processing deals with human language",
    "sentiment analysis classifies text as positive or negative",
    "named entity recognition finds names and places in text",
    "machine translation converts text from one language to another",
    "text summarization produces a shorter version of a document",
    "question answering systems find answers in a passage",
    "word embeddings represent words as dense numeric vectors",
    "the corpus is the collection of text used for training",
    "stopwords are common words that carry little meaning",
    "lemmatization reduces words to their base form",
    "the earth orbits the sun every three hundred sixty five days",
    "water is composed of hydrogen and oxygen atoms",
    "the speed of light is approximately three hundred thousand kilometers per second",
    "gravity attracts objects with mass toward each other",
    "evolution explains the diversity of life on earth",
    "cells are the basic units of all living organisms",
    "photosynthesis converts sunlight into energy for plants",
    "the periodic table organizes chemical elements by atomic number",
    "quantum mechanics describes the behavior of subatomic particles",
    "the human genome contains billions of base pairs",
    "she read a book in the afternoon",
    "he cooked dinner for his family",
    "they played music in the living room",
    "the children went to school in the morning",
    "the train arrived at the station on time",
    "she ordered coffee and a croissant",
    "he walked to the office every morning",
    "the meeting started at nine in the morning",
    "she sent an email to her colleague",
    "the weekend was sunny and warm",
    "they went hiking in the mountains",
    "he bought vegetables at the market",
    "the movie was entertaining and well directed",
    "she learned to play the piano as a child",
    "the concert was held in a large outdoor venue",
    "he repaired the bicycle himself",
    "the library has thousands of books",
    "she finished her thesis after two years",
    "the restaurant was full on saturday evening",
    "he takes the bus to work every day",
]

print(f"{len(corpus)} sentences, {sum(len(s.split()) for s in corpus)} words total")

## 2. Initial vocabulary

Before any merges, every word is represented as a tuple of individual characters plus a special `</w>` end-of-word marker. The marker prevents BPE from accidentally merging the tail of one word with the head of the next.

```python
"the"  (appears 76 times)  →  ('t', 'h', 'e', '</w>') : 76
```

The vocabulary value is the word frequency — how many times it appears in the corpus. This is what makes BPE frequency-weighted: a pair inside a word that appears 76 times contributes 76 to the pair count, not 1.

In [ ]:
def build_vocab(corpus):
    """Build a character-level frequency dictionary from the corpus."""
    vocab = defaultdict(int)
    for sentence in corpus:
        for word in sentence.split():
            vocab[tuple(word) + ('</w>',)] += 1
    return dict(vocab)

vocab = build_vocab(corpus)

print(f"{len(vocab)} unique word forms in the initial vocabulary\n")
print("Top 10 most frequent words:")
for word, freq in sorted(vocab.items(), key=lambda x: -x[1])[:10]:
    print(f"  {freq:3}x  {word}")

## 3. `get_stats(vocab)` — count adjacent pairs

Slide a window of size 2 over every word and count how many times each pair of adjacent symbols appears, weighted by word frequency.

The pair with the highest count is the one BPE merges next.

In [ ]:
def get_stats(vocab):
    """Count every adjacent symbol pair, weighted by word frequency."""
    pairs = defaultdict(int)
    for word, freq in vocab.items():
        for i in range(len(word) - 1):
            pairs[(word[i], word[i + 1])] += freq
    return dict(pairs)

stats = get_stats(vocab)

print(f"{len(stats)} unique pairs found\n")
print(f"  {'PAIR':<25} COUNT")
print(f"  {'-'*35}")
for pair, count in sorted(stats.items(), key=lambda x: -x[1])[:15]:
    print(f"  ('{pair[0]}', '{pair[1]}'){'':<{20 - len(pair[0]) - len(pair[1])}}  {count}")

## 4. `merge_vocab(pair, vocab)` — apply one merge

Takes the winning pair and replaces every consecutive occurrence of it in every word with the fused symbol. Scans left to right — if `('e', '</w>')` wins, every `e` at the end of a word becomes `e</w>` in one pass.

This is called on the single best pair per iteration. After the merge we recompute stats and find the next best pair.

In [ ]:
def merge_vocab(pair, vocab):
    """Replace every occurrence of pair with the merged symbol across the entire vocab."""
    new_vocab = {}
    merged_symbol = pair[0] + pair[1]
    for word, freq in vocab.items():
        new_word = []
        i = 0
        while i < len(word):
            if i < len(word) - 1 and word[i] == pair[0] and word[i + 1] == pair[1]:
                new_word.append(merged_symbol)
                i += 2
            else:
                new_word.append(word[i])
                i += 1
        new_vocab[tuple(new_word)] = freq
    return new_vocab

# Quick demo: merge #1
best_pair = max(stats, key=stats.get)
print(f"Best pair: {best_pair}  (count: {stats[best_pair]})\n")

print("'the' before:", next(w for w in vocab if ''.join(w) == 'the</w>'))
vocab_after_1 = merge_vocab(best_pair, vocab)
print("'the' after :", next(w for w in vocab_after_1 if ''.join(w) == 'the</w>'))

## 5. Training loop — 50 merges

Reset to the initial vocabulary and run 50 iterations. Each iteration finds the best pair, records the merge rule, and applies it.

The `merges` list at the end is everything the tokenizer needs — it's the learned knowledge.

In [ ]:
def get_word_state(vocab, target_word):
    target_str = target_word + '</w>'
    for word in vocab:
        if ''.join(word) == target_str:
            return word
    return None

vocab  = build_vocab(corpus)
merges = []  # ordered merge rules — this IS the tokenizer
tracked = ["the", "training", "model", "learning"]

print("Initial state:")
for w in tracked:
    state = get_word_state(vocab, w)
    print(f"  '{w}' → {state}  [{len(state)} tokens]")
print()

for i in range(50):
    stats     = get_stats(vocab)
    best_pair = max(stats, key=stats.get)
    merged    = best_pair[0] + best_pair[1]
    merges.append((best_pair, merged))
    vocab     = merge_vocab(best_pair, vocab)

    if (i + 1) % 10 == 0:
        print(f"After {i+1} merges  (last: '{best_pair[0]}' + '{best_pair[1]}' → '{merged}'):")
        for w in tracked:
            state = get_word_state(vocab, w)
            print(f"  '{w}' → {state}  [{len(state)} tokens]")
        print()

print(f"Training done. {len(merges)} merge rules learned.")

In [ ]:
# All learned merge rules in order
print(f"{'#':<4} {'PAIR':<35} →  NEW TOKEN")
print("-" * 55)
for idx, (pair, merged) in enumerate(merges):
    print(f"{idx+1:<4} ('{pair[0]}', '{pair[1]}'){'':<{30-len(pair[0])-len(pair[1])}}   '{merged}'")

In [ ]:
# Compression check: how many fewer tokens does the corpus need after 50 merges?
def count_tokens(vocab):
    return sum(len(word) * freq for word, freq in vocab.items())

before = count_tokens(build_vocab(corpus))
after  = count_tokens(vocab)

print(f"Tokens before : {before}")
print(f"Tokens after  : {after}")
print(f"Compression   : {before/after:.2f}x  ({100*(1 - after/before):.1f}% reduction)")

## 6. `encode` and `decode`

The `merges` list defines the tokenizer. To use it on new text:

- **encode**: split into words → characters + `</w>` → apply all 50 merge rules in order → map each remaining symbol to an integer ID
- **decode**: map integers back to symbols → join → replace `</w>` with spaces

IDs are assigned in merge order: base characters get the first IDs (sorted), then each new merged symbol gets the next available ID. The ID number literally tells you when in the training process that token was created.

In [ ]:
def build_token_vocab(corpus, merges):
    """Assign a unique integer to every token (base chars first, then merged symbols)."""
    base_chars = set(ch for sentence in corpus for ch in sentence)
    base_chars.add('</w>')
    token_to_id = {ch: i for i, ch in enumerate(sorted(base_chars))}
    next_id = len(token_to_id)
    for _, merged_symbol in merges:
        if merged_symbol not in token_to_id:
            token_to_id[merged_symbol] = next_id
            next_id += 1
    return token_to_id, {i: t for t, i in token_to_id.items()}

def apply_merges(symbols, merges):
    for pair, merged_symbol in merges:
        i, new_symbols = 0, []
        while i < len(symbols):
            if i < len(symbols)-1 and symbols[i] == pair[0] and symbols[i+1] == pair[1]:
                new_symbols.append(merged_symbol)
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1
        symbols = new_symbols
    return symbols

def encode(text, merges, token_to_id):
    ids = []
    for word in text.lower().split():
        symbols = apply_merges(list(word) + ['</w>'], merges)
        ids.extend(token_to_id[s] for s in symbols)
    return ids

def decode(ids, id_to_token):
    return ''.join(id_to_token[i] for i in ids).replace('</w>', ' ').strip()

token_to_id, id_to_token = build_token_vocab(corpus, merges)
print(f"Vocabulary: {len(token_to_id)} tokens")
print(f"  base characters : {sum(1 for t in token_to_id if len(t) == 1 or t == '</w>')}")
print(f"  merged tokens   : {sum(1 for t in token_to_id if len(t) > 1 and t != '</w>')}")

In [ ]:
# Round-trip verification: decode(encode(text)) == text
test_sentences = [
    "the model learns from training data",
    "attention allows the model to focus on relevant tokens",
    "the cat sat on the mat",
    "gradient descent minimizes the loss function",
    "byte pair encoding merges the most frequent pairs",
]

all_pass = True
for sentence in test_sentences:
    ids  = encode(sentence, merges, token_to_id)
    back = decode(ids, id_to_token)
    ok   = back == sentence
    all_pass = all_pass and ok
    print(f"{'OK' if ok else 'FAIL'}  '{sentence}'")

print(f"\nAll round-trips passed: {all_pass}")

In [ ]:
# Detailed breakdown of a sentence
def show_tokenization(text):
    print(f"'{text}'\n")
    all_ids = []
    for word in text.lower().split():
        symbols = apply_merges(list(word) + ['</w>'], merges)
        ids     = [token_to_id[s] for s in symbols]
        all_ids.extend(ids)
        print(f"  {word:<20} {str(symbols):<45} {ids}")
    print(f"\n  {len(all_ids)} tokens total: {all_ids}")

show_tokenization("the model learns from training data")
print()
show_tokenization("backpropagation computes gradients")

## 7. Comparison with tiktoken (`cl100k_base`)

`cl100k_base` is the tokenizer used by GPT-4. Same BPE algorithm, but trained on hundreds of billions of tokens with ~100k merge rules instead of 50.

I picked 5 sentences to expose specific differences: common words, ML vocabulary, numbers, punctuation, and a rare long word.

In [ ]:
import tiktoken

tik = tiktoken.get_encoding("cl100k_base")
print(f"tiktoken vocab size : {tik.n_vocab:,}")
print(f"our vocab size      : {len(token_to_id)}")

In [ ]:
def our_tokens(text):
    tokens = []
    for word in text.lower().split():
        symbols = apply_merges(list(word) + ['</w>'], merges)
        tokens.extend(s.replace('</w>', '') for s in symbols)
    return [t for t in tokens if t]

def tik_tokens(text):
    return [tik.decode([tid]) for tid in tik.encode(text)]

sentences = [
    "the cat sat on the mat",
    "gradient descent minimizes the loss function",
    "GPT-4 was trained on 1000000 tokens",
    "Hello! What's the temperature today?",
    "bioluminescence is a fascinating phenomenon",
]

for sentence in sentences:
    our = our_tokens(sentence)
    tik_tok = tik_tokens(sentence)
    print(f"'{sentence}'")
    print(f"  ours    ({len(our):2}): {our}")
    print(f"  tiktoken({len(tik_tok):2}): {tik_tok}")
    print()

### Why do they differ?

**Sentence 1 — common English**: both should be efficient on everyday words. `"the"` = 1 token in both. Small differences on short words like `"sat"` or `"on"` where our 50 merges didn't get far enough.

**Sentence 2 — ML vocabulary**: tiktoken was trained on research papers and code, so words like `"gradient"` and `"descent"` appear enough to get merged into single tokens. Ours splits them further because they're rare in our 100-sentence corpus.

**Sentence 3 — numbers and uppercase**: two problems at once. Our tokenizer lowercases everything, so `"GPT"` becomes `"gpt"`. For numbers, BPE has no mathematical notion of digits — `"1000000"` gets split into arbitrary chunks because no number is frequent enough to earn its own token. This is a known weakness of all BPE tokenizers.

**Sentence 4 — punctuation**: our tokenizer crashes on `"!"` and `"?"` because they never appeared in training. tiktoken uses **bytes** as base units (not characters), so it can always represent any Unicode character — `'!'` is just byte 33, guaranteed to exist.

**Sentence 5 — rare long word**: neither has seen `"bioluminescence"` enough to merge it fully. tiktoken does better because it has 100k rules vs our 50 — more sub-word pieces get compressed.

In [ ]:
# tiktoken handles characters our tokenizer can't touch
tricky = ["Hello!", "it's", "GPT-4", "100%", "café", "😀"]

print(f"{'INPUT':<10}  TIKTOKEN")
print("-" * 40)
for word in tricky:
    print(f"  {word:<8}  {tik_tokens(word)}")

print("\nAll of these would crash our tokenizer (unknown characters).")
print("tiktoken handles them because it operates at the byte level, not the character level.")

## Takeaways

The algorithm I implemented here is the same one used by GPT-4. The differences are purely about scale:

- 50 merge rules vs ~100,000
- 100 sentences vs hundreds of billions of tokens  
- characters as base units vs bytes (which guarantees coverage of all Unicode)

One thing that surprised me: the tokenizer has no neural network in it. It's just a sorted list of string replacement rules learned by counting. Once trained, it never changes — the same rules are used for every inference call forever.